# Compound AI patterns — reference solutions

**Solutions for all 18 mini assessments**, matching `compound_ai_pattern_mini_assessments.ipynb`.

Read an exercise solution after attempting it. Each section includes the original problem, unchanged sample data and mock model, a working implementation, an example run, and an explanation of data flow and tradeoffs. These are focused practice solutions, not known Distyl interview answers.

## Running this notebook

Run the shared setup once, then the data, implementation, and example cells for the exercise you want. **Run All also works.** Every example resets the model. Names such as `model` and `TOOLS` are reused, so re-run an exercise’s own data cell before returning to it.

The mocks use recorded responses: successful execution verifies wiring, not real model quality. No API credentials, paid calls, extra packages or external tool execution are needed. Use Python 3.10 or newer. Output cells are intentionally cleared so GitHub does not display every answer before you choose a section.

## Reading the code

`ask` is a convenience function for serializing the supplied payload, calling the mock, and parsing an object. Expand it into `model(...)` and `json.loads(...)` if that makes the data flow clearer. `dispatch_tool` supplies the tool whitelist and argument validation used by T1–T3; its complete implementation appears below. The core pattern lives in each exercise function, while these helpers reduce repeated boilerplate.

The implementations focus on the exercise contracts. Real model APIs need additional schema checks, observability and evidence review; recorded outputs do not establish accuracy or reliable interpretation of user intent.

In [ ]:
import copy
import json
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

class RecordedModel:
    """Thread-safe text-response fixture; required payload keys are checked.

    responses: operation -> list of successive return values
    required: operation -> names of mandatory fields in the JSON prompt
    delays: optional operation -> seconds to simulate a slow call
    Put RuntimeError(...) in a response list to simulate a failed call.
    """
    def __init__(self, responses, required=None, delays=None):
        self._initial = responses
        self.required = required or {}
        self.delays = delays or {}
        self._lock = threading.Lock()
        self.reset()

    def reset(self):
        with self._lock:
            self._responses = copy.deepcopy(self._initial)
            self.calls = []

    def __call__(self, prompt, system_prompt=''):
        op = system_prompt.removeprefix('OP:')
        payload = json.loads(prompt)
        if not isinstance(payload, dict):
            raise ValueError('Prompt must encode a JSON object.')
        missing = set(self.required.get(op, [])) - set(payload)
        if missing:
            raise ValueError(f'{op}: missing payload keys {sorted(missing)}')
        with self._lock:
            if op not in self._responses or not self._responses[op]:
                raise RuntimeError(f'No recorded response left for {op}; inspect the operations or reset the model.')
            response = self._responses[op].pop(0)
            record = {'operation': op, 'payload': copy.deepcopy(payload), 'started': time.perf_counter()}
            self.calls.append(record)
        time.sleep(self.delays.get(op, 0))
        record['finished'] = time.perf_counter()
        if isinstance(response, Exception):
            raise response
        return response if isinstance(response, str) else json.dumps(response)

print('Setup ready. No API keys, external services, or extra packages required.')

In [ ]:
import math


def require(condition, message):
    if not condition:
        raise ValueError(message)


def ask(model, operation, **payload):
    """Shared transport shorthand; returns a parsed JSON object.

    The OP tag belongs to the exercise mock, not to an external model SDK.
    JSON serialization snapshots payloads before later mutations of history.
    """
    raw = model(prompt=json.dumps(payload), system_prompt='OP:' + operation)
    try:
        result = json.loads(raw)
    except (ValueError, TypeError) as error:
        raise ValueError(f'{operation}: expected JSON text') from error
    require(isinstance(result, dict), f'{operation}: expected a JSON object')
    return result


def dispatch_tool(action, tools, tool_descriptions):
    """Shared validation for T1–T3; no eval and no dynamic globals lookup.

    The exercise functions accept only tools present in BOTH supplied maps.
    Type and numeric-bound rules below describe the five local toy tools.
    """
    require(isinstance(action, dict), 'Tool action must be an object')
    name, args = action.get('tool'), action.get('args')
    require(isinstance(name, str) and name in tools and name in tool_descriptions,
            'Unknown tool')
    require(callable(tools[name]), 'Tool is not callable')
    expected = tool_descriptions[name]['args']
    require(isinstance(args, dict) and set(args) == set(expected), 'Incorrect argument keys')
    if name in {'get_stock', 'lookup_account', 'lookup_policy'}:
        key = {'get_stock': 'sku', 'lookup_account': 'account_id', 'lookup_policy': 'topic'}[name]
        require(isinstance(args[key], str) and bool(args[key].strip()), f'{key} must be a nonempty string')
    elif name == 'calculate_quote':
        require(all(type(v) in (int, float) and math.isfinite(v) for v in args.values()),
                'Quote arguments must be finite numbers, not Booleans')
        require(args['list_price'] >= 0 and 0 <= args['discount'] <= 1, 'Invalid price or discount')
    else:
        raise ValueError('No argument-type contract for this tool')
    return tools[name](**args)

## D1 — Event preparation

**Pattern:** Decomposition · **Time:** 10–15 minutes

A team requests an event preparation note covering venue, equipment, and staffing. Use a model-generated task list, obtain an answer for each task, then produce one combined note. Keep this first drill sequential; concurrency is reserved for P1–P3. Return `{"tasks": [...], "answers": {...}, "final_note": "..."}`.

**Supplied model operations**

- `plan`: payload `{request}` → `{tasks: [{id, question}]}`.
- `answer_venue`, `answer_equipment`, `answer_staffing`: payload `{request, task}` → `{answer}`. Choose the operation from the task ID, not list position.
- `combine`: payload `{request, answers}` → `{final_note}`.

**Manual success criteria**

- Use five calls on this fixture: plan, three task answers, and combination.
- Store answers under task IDs and pass all three answers to `combine`.
- Read the task list from the model response rather than hardcoding the three answers.

In [ ]:
REQUEST = "Prepare a note for a 12-person workshop: venue, equipment, and staffing."
model = RecordedModel({
 'plan': [{'tasks': [{'id':'venue','question':'What venue is needed?'}, {'id':'equipment','question':'What equipment is needed?'}, {'id':'staffing','question':'Who should support the event?'}]}],
 'answer_venue': [{'answer':'A room with 12 seats and accessible entry.'}],
 'answer_equipment': [{'answer':'One projector and a shared whiteboard.'}],
 'answer_staffing': [{'answer':'One facilitator and one check-in helper.'}],
 'combine': [{'final_note':'Reserve a 12-seat accessible room, prepare a projector and whiteboard, and assign a facilitator and check-in helper.'}]
}, required={'plan':['request'], **{f'answer_{x}':['request','task'] for x in ['venue','equipment','staffing']}, 'combine':['request','answers']})

### Worked approach

Plan → solve each task → combine. Task IDs keep the intermediate answers associated with their original questions.

In [ ]:
def prepare_event(request, model):
    plan = ask(model, 'plan', request=request)
    tasks = plan['tasks']
    answers = {}
    for task in tasks:
        task_id = task['id']
        require(task_id not in answers, 'Duplicate task ID')
        reply = ask(model, f'answer_{task_id}', request=request, task=task)
        answers[task_id] = reply['answer']
    final = ask(model, 'combine', request=request, answers=answers)
    return {'tasks': tasks, 'answers': answers, 'final_note': final['final_note']}

In [ ]:
model.reset()
result = prepare_event(REQUEST, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

The combining call receives all three actual answers. A reordered task list should still work. Five calls increase cost; decomposition helps only if focused subtasks improve coverage.

## D2 — Compare by criterion

**Pattern:** Decomposition · **Time:** 10–15 minutes

Compare two devices on battery life and portability. The model selects criteria. Analyze each criterion against the supplied facts, then assemble a comparison keyed by criterion. The combination step may be Python; no synthesis call is required. Return a list of `{criterion, comparison, preferred}` rows.

**Supplied model operations**

- `criteria`: payload `{request}` → `{criteria: [string]}`.
- `compare_battery`, `compare_portability`: payload `{criterion, products}` → `{comparison, preferred}`.

**Manual success criteria**

- Obtain criteria from the planning call and process every criterion.
- Supply both product records to each comparison call.
- Preserve the criterion with each result; do not invent an overall winner.

In [ ]:
REQUEST = "Compare Atlas and Birch for commuting, considering battery and portability."
PRODUCTS = [{'name':'Atlas','battery_hours':11,'weight_kg':1.8}, {'name':'Birch','battery_hours':8,'weight_kg':1.2}]
model = RecordedModel({
 'criteria':[{'criteria':['battery','portability']}],
 'compare_battery':[{'comparison':'Atlas lasts longer: 11 versus 8 hours.','preferred':'Atlas'}],
 'compare_portability':[{'comparison':'Birch is lighter: 1.2 versus 1.8 kg.','preferred':'Birch'}]
}, required={'criteria':['request'], 'compare_battery':['criterion','products'], 'compare_portability':['criterion','products']})

### Worked approach

Select criteria → compare both products for each criterion. Python assembles the rows; another model call is unnecessary for this output.

In [ ]:
def compare_products(request, products, model):
    plan = ask(model, 'criteria', request=request)
    rows = []
    for criterion in plan['criteria']:
        reply = ask(model, f'compare_{criterion}', criterion=criterion, products=products)
        rows.append({'criterion': criterion,
                     'comparison': reply['comparison'], 'preferred': reply['preferred']})
    return rows

In [ ]:
model.reset()
result = compare_products(REQUEST, PRODUCTS, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

Each comparison sees both product records. Preserve separate preferences instead of inventing a global winner. The mock operation names only support the criteria in this drill.

## D3 — Respect task dependencies

**Pattern:** Decomposition · **Time:** 10–15 minutes

Prepare an onboarding plan where account setup depends on an approved role, and training depends on account setup. The task list is deliberately unordered. Execute only tasks whose dependencies have completed, passing their results as context. Return results keyed by task ID. If a pass makes no progress, raise `ValueError` for an unresolved dependency or cycle.

**Supplied model operations**

- `plan`: payload `{request}` → `{tasks: [{id, depends_on}]}`.
- `do_role`, `do_account`, `do_training`: payload `{request, task, dependency_results}` → `{result}`.

**Manual success criteria**

- Complete role before account and account before training despite the supplied order.
- Pass actual dependency outputs to later calls.
- Explain how your no-progress check handles a cycle; do not run a cyclic loop indefinitely.

In [ ]:
REQUEST = "Plan onboarding for a new analyst."
model = RecordedModel({
 'plan':[{'tasks':[{'id':'training','depends_on':['account']}, {'id':'role','depends_on':[]}, {'id':'account','depends_on':['role']}]}],
 'do_role':[{'result':'Approved role: analyst.'}],
 'do_account':[{'result':'Analyst account prepared.'}],
 'do_training':[{'result':'Account-specific training scheduled.'}]
}, required={'plan':['request'], **{f'do_{x}':['request','task','dependency_results'] for x in ['role','account','training']}})

### Worked approach

Repeatedly run tasks whose dependency results are available. A no-progress pass means a missing dependency or cycle.

In [ ]:
def plan_onboarding(request, model):
    tasks = ask(model, 'plan', request=request)['tasks']
    require(isinstance(tasks, list), 'Task list required')
    task_ids = [task['id'] for task in tasks]
    require(len(task_ids) == len(set(task_ids)), 'Duplicate task IDs')
    for task in tasks:
        require(isinstance(task['depends_on'], list), 'Dependency list required')
    pending = list(tasks)
    results = {}
    while pending:
        progressed = False
        for task in pending[:]:
            deps = task['depends_on']
            if not all(dep in results for dep in deps):
                continue
            context = {dep: results[dep] for dep in deps}
            reply = ask(model, f"do_{task['id']}", request=request,
                        task=task, dependency_results=context)
            results[task['id']] = reply['result']
            pending.remove(task)
            progressed = True
        if not progressed:
            raise ValueError('Unresolved dependency or cycle: ' +
                             ', '.join(task['id'] for task in pending))
    return results

In [ ]:
model.reset()
result = plan_onboarding(REQUEST, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

The account task receives the approved-role result, and training receives the account result. The list order does not determine execution. More general scheduling can use a topological sort.

## R1 — Repair an incomplete notice

**Pattern:** Refinement · **Time:** 10–15 minutes

Draft an office notice that must include a date, time, and location. Ask for feedback, then request one revision using that feedback. Return `{draft, feedback, revised}`. This drill has exactly one review and one revision.

**Supplied model operations**

- `draft`: payload `{facts}` → `{text}`.
- `review`: payload `{facts, draft}` → `{issues: [string]}`.
- `revise`: payload `{facts, draft, issues}` → `{text}`.

**Manual success criteria**

- The review receives the actual initial text and original facts.
- The revision receives both the draft and the review issues.
- Keep the initial draft available so you can compare versions.

In [ ]:
FACTS = {'date':'June 12','time':'14:00','location':'Room Cedar','purpose':'Team workshop'}
model = RecordedModel({
 'draft':[{'text':'Join our team workshop at 14:00 in Room Cedar.'}],
 'review':[{'issues':['The notice omits June 12.']}],
 'revise':[{'text':'Join our team workshop on June 12 at 14:00 in Room Cedar.'}]
}, required={'draft':['facts'], 'review':['facts','draft'], 'revise':['facts','draft','issues']})

### Worked approach

Draft → critique → revise once. Preserve the initial version so changes can be inspected.

In [ ]:
def refine_notice(facts, model):
    draft = ask(model, 'draft', facts=facts)['text']
    feedback = ask(model, 'review', facts=facts, draft=draft)['issues']
    revised = ask(model, 'revise', facts=facts, draft=draft, issues=feedback)['text']
    return {'draft': draft, 'feedback': feedback, 'revised': revised}

In [ ]:
model.reset()
result = refine_notice(FACTS, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

The revision sees the original facts, draft, and feedback. Check that the missing date was added and the correct time/location were retained. This fixed drill always revises; R3 makes revision conditional.

## R2 — Repair using Python feedback

**Pattern:** Refinement · **Time:** 10–15 minutes

Extract a product and quantity. The first response has the wrong type for quantity. Validate that the output is a dictionary with a nonempty string `product` and a nonnegative integer `quantity` (a Boolean does not count). On failure, make one repair call with the raw response and specific error. Return a validated dictionary or raise `ValueError`; do not loop indefinitely.

**Supplied model operations**

- `extract`: payload `{text}` → JSON text.
- `repair`: payload `{text, previous_output, error}` → corrected JSON text.

**Manual success criteria**

- Reject string and Boolean quantities rather than coercing them silently.
- Report the actual failed requirement in the repair payload.
- Allow at most two calls total. Revalidate the repair before returning.

In [ ]:
TEXT = "Please order four desk lamps."
model = RecordedModel({
 'extract':['{"product":"desk lamp","quantity":"four"}'],
 'repair':[{'product':'desk lamp','quantity':4}]
}, required={'extract':['text'], 'repair':['text','previous_output','error']})

### Worked approach

Generate → deterministic validation → one repair if needed → revalidate.

In [ ]:
def extract_with_repair(text, model):
    def validate(raw):
        try:
            data = json.loads(raw)
        except (ValueError, TypeError) as exc:
            raise ValueError('Response must be valid JSON') from exc
        require(isinstance(data, dict), 'Response must be an object')
        require(isinstance(data.get('product'), str) and bool(data['product'].strip()),
                'product must be a nonempty string')
        require(type(data.get('quantity')) is int and data['quantity'] >= 0,
                'quantity must be a nonnegative integer; bool is not allowed')
        return data

    raw = model(json.dumps({'text': text}), system_prompt='OP:extract')
    try:
        return validate(raw)
    except ValueError as error:
        repaired = model(json.dumps({'text': text, 'previous_output': raw,
                                     'error': str(error)}), system_prompt='OP:repair')
        return validate(repaired)  # A second failure propagates; no third call.

In [ ]:
model.reset()
result = extract_with_repair(TEXT, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

The repair receives the raw response and a specific error. A Boolean is excluded with type(value) is int. Invalid repaired output raises rather than silently becoming an empty successful extraction.

## R3 — Stop a revision loop

**Pattern:** Refinement · **Time:** 10–15 minutes

Produce a handoff note, review it, and revise only while the reviewer finds problems. Allow at most two revisions. Stop immediately when `passed` is true. Return `{text, revisions, passed}`; if still failing at the limit, return `passed=false` with the last text.

**Supplied model operations**

- `draft`: payload `{request}` → `{text}`.
- `review`: payload `{request, text}` → `{passed: boolean, issues: [string]}`.
- `revise`: payload `{request, text, issues}` → `{text}`.

**Manual success criteria**

- This fixture needs four calls and one revision; no second revision should happen.
- Count revisions separately from review calls.
- The cap applies even if a reviewer never approves the output.

In [ ]:
REQUEST = "Create a handoff note with an owner and next action."
model = RecordedModel({
 'draft':[{'text':'Follow up on the migration.'}],
 'review':[{'passed':False,'issues':['Missing owner.']}, {'passed':True,'issues':[]}],
 'revise':[{'text':'Owner: Priya. Next action: follow up on the migration.'}]
}, required={'draft':['request'], 'review':['request','text'], 'revise':['request','text','issues']})

### Worked approach

Generate once; review the current version; revise only if review fails and a revision budget remains.

In [ ]:
def bounded_refinement(request, model, max_revisions=2):
    require(type(max_revisions) is int and max_revisions >= 0, 'Invalid revision limit')
    text = ask(model, 'draft', request=request)['text']
    revisions = 0
    while True:
        review = ask(model, 'review', request=request, text=text)
        require(type(review.get('passed')) is bool, 'passed must be a Boolean')
        require(isinstance(review.get('issues'), list), 'issues must be a list')
        if review['passed'] or revisions == max_revisions:
            return {'text': text, 'revisions': revisions, 'passed': review['passed']}
        text = ask(model, 'revise', request=request, text=text, issues=review['issues'])['text']
        revisions += 1

In [ ]:
model.reset()
result = bounded_refinement(REQUEST, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

Review calls and revision calls are counted separately. At a limit of two revisions there can be three reviews. A passed flag is model feedback, not proof of correctness.

## F1 — Resolve references from context

**Pattern:** Reformulation · **Time:** 10–15 minutes

Rewrite the follow-up question as a standalone question using the conversation history. Then pass that standalone question to an answering call. Return `{standalone_question, answer}`. Preserve the time period and entities from the supplied context.

**Supplied model operations**

- `rewrite`: payload `{history, follow_up}` → `{standalone_question}`.
- `answer`: payload `{question, facts}` → `{answer}`.

**Manual success criteria**

- The second call receives the rewritten question, not the original ambiguous follow-up.
- Supply the facts to the answering call; rewriting alone does not provide evidence.
- Keep the standalone question in the output for inspection.

In [ ]:
HISTORY = [{'role':'user','text':'Compare the Q1 and Q2 support backlogs.'}, {'role':'assistant','text':'The backlog was 120 tickets in Q1 and 90 in Q2.'}]
FOLLOW_UP = "How much did it drop?"
FACTS = {'Q1_backlog':120,'Q2_backlog':90}
model = RecordedModel({
 'rewrite':[{'standalone_question':'How many support tickets did the backlog decrease by from Q1 to Q2?'}],
 'answer':[{'answer':'The support backlog decreased by 30 tickets.'}]
}, required={'rewrite':['history','follow_up'], 'answer':['question','facts']})

### Worked approach

Resolve the follow-up against history, then answer the standalone question using the supplied facts.

In [ ]:
def answer_follow_up(history, follow_up, facts, model):
    question = ask(model, 'rewrite', history=history, follow_up=follow_up)['standalone_question']
    require(isinstance(question, str) and bool(question.strip()), 'Empty rewritten question')
    answer = ask(model, 'answer', question=question, facts=facts)['answer']
    return {'standalone_question': question, 'answer': answer}

In [ ]:
model.reset()
result = answer_follow_up(HISTORY, FOLLOW_UP, FACTS, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

The downstream answering call receives the rewritten question and factual data, not an ambiguous pronoun. In real use, inspect that rewriting preserved time periods, entities, and user intent.

## F2 — Create useful search queries

**Pattern:** Reformulation · **Time:** 10–15 minutes

Turn a vague request into precise search queries, normalize whitespace, remove exact duplicates, and keep at most three. Return `{queries, results}` using the supplied deterministic search tool. Do not write an answer to the request in this drill.

**Supplied model operations**

- `rewrite_queries`: payload `{request}` → `{queries: [string]}`.
- `search(query)` is supplied Python infrastructure, not a model call.

**Manual success criteria**

- Return three distinct nonempty queries in their original order after normalization.
- Call `search` once per retained query and preserve the query/result association.
- Do not silently add a guessed date, system name, or technology.

In [ ]:
REQUEST = "Find notes about our old login setup and why we changed it."
SEARCH_INDEX = {
 'legacy authentication migration rationale':['DOC-11'],
 'previous login architecture':['DOC-12'],
 'authentication migration decision record':['DOC-13']
}
def search(query):
    return SEARCH_INDEX.get(query, [])
model = RecordedModel({
 'rewrite_queries':[{'queries':[' legacy authentication migration rationale ', 'previous login architecture', 'legacy authentication migration rationale', 'authentication migration decision record']}]
}, required={'rewrite_queries':['request']})

### Worked approach

Rewrite into precise queries, normalize whitespace, deduplicate while preserving order, then look up each retained query.

In [ ]:
def reformulate_and_search(request, model, search):
    raw_queries = ask(model, 'rewrite_queries', request=request)['queries']
    require(isinstance(raw_queries, list), 'queries must be a list')
    queries = []
    seen = set()
    for raw in raw_queries:
        require(isinstance(raw, str), 'Every query must be a string')
        query = ' '.join(raw.split())
        if query and query not in seen:
            queries.append(query)
            seen.add(query)
        if len(queries) == 3:
            break
    results = {query: search(query) for query in queries}
    return {'queries': queries, 'results': results}

In [ ]:
model.reset()
result = reformulate_and_search(REQUEST, model, search)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

The result dictionary records which query produced each document ID. Duplicate queries are removed before retrieval, avoiding redundant search. Search results are not yet an answer.

## F3 — Keep ambiguity visible

**Pattern:** Reformulation · **Time:** 10–15 minutes

Reformulate a request with an ambiguous pronoun. The model returns candidate interpretations and whether clarification is needed. If clarification is needed, return `{status: "needs_clarification", question, interpretations}` and do not call `execute_request`. Otherwise return `{status: "ready", request}` from the sole interpretation. This exercise proposes a clarified request; it sends nothing.

**Supplied model operations**

- `interpret`: payload `{request, context}` → `{needs_clarification, interpretations, clarification_question}`.
- `execute_request` is a supplied recording function; it must not run on the ambiguous fixture.

**Manual success criteria**

- Preserve both possible interpretations and return the blocking question.
- `EXECUTIONS` remains empty for this fixture.
- Explain how your ready branch rejects zero or multiple interpretations rather than selecting arbitrarily.

In [ ]:
REQUEST = "Send it to Jordan after the review."
CONTEXT = {'documents':['budget memo','vendor summary'], 'people':['Jordan Lee','Jordan Patel']}
EXECUTIONS = []
def execute_request(request):
    EXECUTIONS.append(request)
    return request
model = RecordedModel({
 'interpret':[{'needs_clarification':True,'interpretations':['Send the budget memo to Jordan Lee.','Send the vendor summary to Jordan Patel.'], 'clarification_question':'Which document and which Jordan do you mean?'}]
}, required={'interpret':['request','context']})

### Worked approach

Keep competing interpretations visible and ask a blocking clarification. Only a sole unambiguous interpretation can enter the ready branch.

In [ ]:
def clarify_request(request, context, model, execute_request):
    reply = ask(model, 'interpret', request=request, context=context)
    require(type(reply.get('needs_clarification')) is bool, 'Expected Boolean flag')
    interpretations = reply.get('interpretations')
    require(isinstance(interpretations, list) and
            all(isinstance(x, str) and x.strip() for x in interpretations),
            'Interpretations must be nonempty strings')
    if reply['needs_clarification']:
        question = reply.get('clarification_question')
        require(isinstance(question, str) and bool(question.strip()), 'Missing clarification')
        return {'status': 'needs_clarification', 'question': question,
                'interpretations': interpretations}
    require(len(interpretations) == 1, 'Ready requests must have exactly one interpretation')
    ready = interpretations[0]
    execute_request(ready)  # A local recording function only; no real message is sent.
    return {'status': 'ready', 'request': ready}

In [ ]:
model.reset()
EXECUTIONS.clear()
result = clarify_request(REQUEST, CONTEXT, model, execute_request)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

The ambiguous fixture never calls the recording function. The ready branch records a proposed request locally, as specified in the supplied exercise; it sends no real message. Real ambiguity detection remains a semantic judgment.

## G1 — Choose one specialist

**Pattern:** Routing · **Time:** 10–15 minutes

Classify a support message and send it to exactly one specialist. Allowed routes are `billing`, `technical`, and `general`. Reject an unknown route with `ValueError`. Return `{route, response}`.

**Supplied model operations**

- `classify`: payload `{message}` → `{route}`.
- `billing`, `technical`, `general`: payload `{message}` → `{response}`. Only the selected operation should be called.

**Manual success criteria**

- Make one classification call and one billing call on the sample.
- Pass the original message to the specialist.
- Leave unselected specialist responses unused.

In [ ]:
MESSAGE = "I was charged twice for my subscription."
model = RecordedModel({
 'classify':[{'route':'billing'}],
 'billing':[{'response':'Please provide the two transaction references so Billing can investigate.'}],
 'technical':[{'response':'Technical support response.'}],
 'general':[{'response':'General support response.'}]
}, required={x:['message'] for x in ['classify','billing','technical','general']})

### Worked approach

Classify first, validate the route, then call exactly one specialist.

In [ ]:
def route_message(message, model):
    route = ask(model, 'classify', message=message)['route']
    require(isinstance(route, str) and route in {'billing', 'technical', 'general'},
            'Unknown route')
    response = ask(model, route, message=message)['response']
    return {'route': route, 'response': response}

In [ ]:
model.reset()
result = route_message(MESSAGE, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

Passing the original message preserves context. Unknown routes fail before a specialist is invoked. This chooses one path; it does not solve multiple subtasks.

## G2 — Route a batch without losing IDs

**Pattern:** Routing · **Time:** 10–15 minutes

Classify two tickets in one model call, then invoke the selected specialist separately for each ticket. Routing rows may arrive out of order. Return results in the original ticket order as `{ticket_id, route, response}` rows. Validate that every ticket has exactly one known route.

**Supplied model operations**

- `classify_batch`: payload `{tickets}` → `{routes: [{ticket_id, route}]}`.
- `billing`, `technical`, `general`: payload `{ticket}` → `{response}`.

**Manual success criteria**

- Match routing rows by ticket ID, not position.
- Return A then B even though classification returns B then A.
- Reject missing, duplicate, or invented ticket IDs.

In [ ]:
TICKETS = [{'ticket_id':'A','message':'I cannot log in.'}, {'ticket_id':'B','message':'My invoice total is wrong.'}]
model = RecordedModel({
 'classify_batch':[{'routes':[{'ticket_id':'B','route':'billing'},{'ticket_id':'A','route':'technical'}]}],
 'technical':[{'response':'Check the sign-in error and account status.'}],
 'billing':[{'response':'Review the invoice line items.'}],
 'general':[{'response':'General support response.'}]
}, required={'classify_batch':['tickets'], **{x:['ticket'] for x in ['billing','technical','general']}})

### Worked approach

Classify as a batch, validate a complete ID-to-route mapping, and dispatch in the input order.

In [ ]:
def route_batch(tickets, model):
    ids = [ticket['ticket_id'] for ticket in tickets]
    require(len(ids) == len(set(ids)), 'Duplicate input ticket IDs')
    rows = ask(model, 'classify_batch', tickets=tickets)['routes']
    require(isinstance(rows, list), 'routes must be a list')
    routes = {}
    for row in rows:
        ticket_id, route = row['ticket_id'], row['route']
        require(ticket_id in ids and ticket_id not in routes, 'Unknown or duplicate ticket ID')
        require(isinstance(route, str) and route in {'billing', 'technical', 'general'},
                'Unknown specialist route')
        routes[ticket_id] = route
    require(set(routes) == set(ids), 'Missing ticket route')
    results = []
    for ticket in tickets:  # Original order, independent of classification order.
        route = routes[ticket['ticket_id']]
        reply = ask(model, route, ticket=ticket)
        results.append({'ticket_id': ticket['ticket_id'], 'route': route,
                        'response': reply['response']})
    return results

In [ ]:
model.reset()
result = route_batch(TICKETS, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

Never zip returned rows to tickets: the model may reorder them. Validate the entire mapping before any specialist runs, preventing partial dispatch when routing data is incomplete.

## G3 — Fall back when routing is uncertain

**Pattern:** Routing · **Time:** 10–15 minutes

Route a message only when the returned route is allowed and confidence is at least 0.8. Otherwise return `{route: "human_review", reason, response: null}` without a specialist call. Do not treat model confidence as a calibrated probability; the threshold is merely this drill’s dispatch rule.

**Supplied model operations**

- `classify`: payload `{message}` → `{route, confidence, reason}`.
- Specialist operations have the same interface as G1; the uncertain fixture should call none.

**Manual success criteria**

- This fixture returns human_review after one call.
- Unknown routes and low confidence both trigger fallback.
- Validate confidence as a numeric value between zero and one; reject malformed types.

In [ ]:
MESSAGE = "The charge failed, but money disappeared and the app froze."
model = RecordedModel({
 'classify':[{'route':'technical','confidence':0.55,'reason':'Both payment and application failure may be involved.'}],
 'technical':[{'response':'A technical-only answer.'}]
}, required={'classify':['message'],'technical':['message']})

### Worked approach

Validate the numeric confidence, then either dispatch a known route or return human review.

In [ ]:
def route_with_fallback(message, model, threshold=0.8):
    require(type(threshold) in (int, float) and math.isfinite(threshold) and
            0 <= threshold <= 1, 'Invalid routing threshold')
    reply = ask(model, 'classify', message=message)
    confidence = reply.get('confidence')
    require(type(confidence) in (int, float) and math.isfinite(confidence) and
            0 <= confidence <= 1, 'Confidence must be a finite number between 0 and 1')
    route = reply.get('route')
    if not isinstance(route, str) or route not in {'billing', 'technical', 'general'}:
        return {'route': 'human_review', 'reason': 'Unknown route', 'response': None}
    if confidence < threshold:
        return {'route': 'human_review', 'reason': reply.get('reason', 'Low confidence'),
                'response': None}
    response = ask(model, route, message=message)['response']
    return {'route': route, 'response': response}

In [ ]:
model.reset()
result = route_with_fallback(MESSAGE, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

Unknown labels and low scores do not trigger a specialist. Reject strings, Booleans, non-finite numbers and out-of-range scores. The chosen threshold is an exercise rule, not a calibrated probability.

## P1 — Summarize independent notes

**Pattern:** Parallelization · **Time:** 10–15 minutes

Summarize three independent notes concurrently with `ThreadPoolExecutor`, using three workers. Collect results keyed by note ID, regardless of completion order. Return `{summaries, elapsed_seconds}`. Measure the call group, not notebook setup.

**Supplied model operations**

- `summary_A`, `summary_B`, `summary_C`: payload `{note}` → `{summary}`.
- Simulated delays differ intentionally. `model.calls` records start and finish times.

**Manual success criteria**

- Three calls overlap; inspect timestamps rather than relying on a strict timing assertion.
- All IDs map to the correct summaries, even though B finishes before A.
- Do not call `.result()` immediately after submitting each individual task; that would serialize the work.

In [ ]:
NOTES = [{'id':'A','text':'The support backlog shrank.'}, {'id':'B','text':'The new release shipped.'}, {'id':'C','text':'Hiring is paused.'}]
model = RecordedModel({
 'summary_A':[{'summary':'Support backlog decreased.'}],
 'summary_B':[{'summary':'Release delivered.'}],
 'summary_C':[{'summary':'Hiring paused.'}]
}, required={f'summary_{x}':['note'] for x in 'ABC'}, delays={'summary_A':0.30,'summary_B':0.10,'summary_C':0.20})

### Worked approach

Submit all independent summaries before waiting for results; associate each future with its note ID.

In [ ]:
def parallel_summaries(notes, model):
    ids = [note['id'] for note in notes]
    require(len(ids) == len(set(ids)), 'Duplicate note IDs')
    started = time.perf_counter()
    summaries = {}
    with ThreadPoolExecutor(max_workers=3) as pool:
        futures = {
            pool.submit(ask, model, f"summary_{note['id']}", note=note): note['id']
            for note in notes
        }
        for future in as_completed(futures):
            summaries[futures[future]] = future.result()['summary']
    return {'summaries': summaries, 'elapsed_seconds': time.perf_counter() - started}

In [ ]:
model.reset()
result = parallel_summaries(NOTES, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])
print('Call intervals:', [(call['operation'], round(call['started'] - model.calls[0]['started'], 3), round(call['finished'] - model.calls[0]['started'], 3)) for call in model.calls])

### Why it works and what to watch

The sample calls should overlap. Collection order is nondeterministic, but result attribution is stable. Concurrency reduces waiting time, not the number of paid model calls.

## P2 — Independent perspectives, then synthesis

**Pattern:** Parallelization · **Time:** 10–15 minutes

Ask an operations reviewer and a finance reviewer to examine the same proposal independently and concurrently. After both finish, make one synthesis call using both reviews. Return `{reviews, decision}`. Neither reviewer should see the other reviewer’s output.

**Supplied model operations**

- `ops_review`, `finance_review`: payload `{proposal}` → `{findings}`.
- `synthesize`: payload `{proposal, reviews}` → `{decision}`.

**Manual success criteria**

- Review calls overlap, and synthesis starts after both finish.
- Preserve reviewer labels in the synthesis payload.
- Do not present agreement between reviewers as proof the decision is correct.

In [ ]:
PROPOSAL = "Add weekend coverage for a three-month pilot."
model = RecordedModel({
 'ops_review':[{'findings':['Coverage may reduce weekend backlog.','Staff availability is unknown.']}],
 'finance_review':[{'findings':['Pilot cost needs an explicit budget.']}],
 'synthesize':[{'decision':'Define staffing availability and a capped budget before starting the pilot.'}]
}, required={'ops_review':['proposal'],'finance_review':['proposal'],'synthesize':['proposal','reviews']}, delays={'ops_review':0.20,'finance_review':0.20})

### Worked approach

Run two independent reviews concurrently, then wait for both before synthesis.

In [ ]:
def parallel_reviews(proposal, model):
    reviews = {}
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = {pool.submit(ask, model, op, proposal=proposal): op
                   for op in ('ops_review', 'finance_review')}
        for future in as_completed(futures):
            reviews[futures[future]] = future.result()['findings']
    decision = ask(model, 'synthesize', proposal=proposal, reviews=reviews)['decision']
    return {'reviews': reviews, 'decision': decision}

In [ ]:
model.reset()
result = parallel_reviews(PROPOSAL, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])
print('Call intervals:', [(call['operation'], round(call['started'] - model.calls[0]['started'], 3), round(call['finished'] - model.calls[0]['started'], 3)) for call in model.calls])

### Why it works and what to watch

Reviewers see only the proposal. Synthesis receives both labeled results. Independence here means neither sees the other response; use of the same underlying model does not imply independent errors.

## P3 — Preserve partial success

**Pattern:** Parallelization · **Time:** 10–15 minutes

Process three independent documents concurrently. One model call fails. Return `{results, errors}` keyed by document ID, retaining successful results rather than discarding the entire batch. Do not retry or synthesize in this drill; isolate concurrent collection and error handling.

**Supplied model operations**

- `extract_A`, `extract_B`, `extract_C`: payload `{document}` → `{fields}` or an exception.
- The B fixture raises `RuntimeError` intentionally.

**Manual success criteria**

- Keep A and C in results; put B in errors.
- Associate every future with its document ID before collecting it.
- A failed call must not become an empty successful extraction.

In [ ]:
DOCUMENTS = [{'id':'A','text':'Owner: Morgan'}, {'id':'B','text':'Unreadable document'}, {'id':'C','text':'Owner: Taylor'}]
model = RecordedModel({
 'extract_A':[{'fields':{'owner':'Morgan'}}],
 'extract_B':[RuntimeError('Simulated model timeout')],
 'extract_C':[{'fields':{'owner':'Taylor'}}]
}, required={f'extract_{x}':['document'] for x in 'ABC'}, delays={'extract_A':0.20,'extract_B':0.10,'extract_C':0.15})

### Worked approach

Run all independent extractions and handle failure separately for each future.

In [ ]:
def parallel_extract(documents, model):
    ids = [document['id'] for document in documents]
    require(len(ids) == len(set(ids)), 'Duplicate document IDs')
    results, errors = {}, {}
    with ThreadPoolExecutor(max_workers=3) as pool:
        futures = {
            pool.submit(ask, model, f"extract_{doc['id']}", document=doc): doc['id']
            for doc in documents
        }
        for future in as_completed(futures):
            doc_id = futures[future]
            try:
                results[doc_id] = future.result()['fields']
            except Exception as error:
                errors[doc_id] = f'{type(error).__name__}: {error}'
    return {'results': results, 'errors': errors}

In [ ]:
model.reset()
result = parallel_extract(DOCUMENTS, model)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])
print('Call intervals:', [(call['operation'], round(call['started'] - model.calls[0]['started'], 3), round(call['finished'] - model.calls[0]['started'], 3)) for call in model.calls])

### Why it works and what to watch

Successful A and C results remain available when B fails. Catching errors at collection preserves attribution. Broader exception catching is deliberate at this batch boundary; do not turn exceptions into empty successful fields.

## T1 — Execute one local lookup

**Pattern:** Tool use · **Time:** 10–15 minutes

Answer a stock question using the supplied `get_stock` tool. First ask the model what tool to call, validate the name and exact argument keys, run the Python function, then send its result to the model for a final answer. Return `{tool_call, tool_result, answer}`. Never use `eval` or dispatch arbitrary names from globals.

**Supplied model operations**

- `decide`: payload `{question, tools}` → `{tool, args}`.
- `respond`: payload `{question, tool_call, tool_result}` → `{answer}`.
- Allowed tool: `get_stock(sku)`; it returns local inventory data.

**Manual success criteria**

- Execute the actual supplied function and pass its return value to `respond`.
- Reject unknown tools, missing/extra argument keys, and non-string SKUs.
- Do not hardcode the final inventory answer or let the model invent stock data.

In [ ]:
QUESTION = "How many KIT-7 units are available?"
INVENTORY = {'KIT-7':{'available':8},'KIT-9':{'available':0}}
def get_stock(sku):
    if sku not in INVENTORY:
        raise ValueError('Unknown SKU')
    return INVENTORY[sku]
TOOLS = {'get_stock': get_stock}
TOOL_DESCRIPTIONS = {'get_stock':{'args':['sku'],'description':'Look up available units for a SKU.'}}
model = RecordedModel({
 'decide':[{'tool':'get_stock','args':{'sku':'KIT-7'}}],
 'respond':[{'answer':'There are 8 KIT-7 units available.'}]
}, required={'decide':['question','tools'],'respond':['question','tool_call','tool_result']})

### Worked approach

Ask for an action → validate and execute a local tool → pass the observation to the model for an answer.

In [ ]:
def answer_stock_question(question, model, tools, tool_descriptions):
    action = ask(model, 'decide', question=question, tools=tool_descriptions)
    result = dispatch_tool(action, tools, tool_descriptions)
    answer = ask(model, 'respond', question=question,
                 tool_call=action, tool_result=result)['answer']
    return {'tool_call': action, 'tool_result': result, 'answer': answer}

In [ ]:
model.reset()
result = answer_stock_question(QUESTION, model, TOOLS, TOOL_DESCRIPTIONS)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

The function result, not the model’s guessed inventory, is the source for the final call. dispatch_tool is defined in the shared setup; it whitelists names and validates argument names and types.

## T2 — Continue after a tool result

**Pattern:** Tool use · **Time:** 10–15 minutes

Run a bounded tool-use loop. The model may request a local account lookup, then a quote calculation using the returned discount, then provide a final answer. Append every action and observation to history. Allow at most four model calls, and raise `RuntimeError` if no final answer appears. Return `{answer, history}`.

**Supplied model operations**

- `next`: payload `{question, tools, history}` → either `{type:"tool", tool, args}` or `{type:"final", answer}`.
- Allowed tools: `lookup_account(account_id)` and `calculate_quote(list_price, discount)`.
- This fixture needs three model calls and two local tool calls.

**Manual success criteria**

- The second model call sees the account lookup observation; the final call sees the quote result.
- Validate tool names, exact argument keys and basic argument types before dispatch.
- Do not treat numeric Booleans as prices; require a nonnegative list price and discount between zero and one.
- Stop when the model returns a final answer or the model-call budget is exhausted.

In [ ]:
QUESTION = "What is the quote for account C-1 on an item listed at 100?"
def lookup_account(account_id):
    if account_id != 'C-1':
        raise ValueError('Unknown account')
    return {'discount':0.10}
def calculate_quote(list_price, discount):
    return {'total':round(list_price * (1 - discount), 2)}
TOOLS = {'lookup_account':lookup_account, 'calculate_quote':calculate_quote}
TOOL_DESCRIPTIONS = {'lookup_account':{'args':['account_id']}, 'calculate_quote':{'args':['list_price','discount']}}
model = RecordedModel({
 'next':[{'type':'tool','tool':'lookup_account','args':{'account_id':'C-1'}},
         {'type':'tool','tool':'calculate_quote','args':{'list_price':100,'discount':0.10}},
         {'type':'final','answer':'The quote is 90.00 after the account discount.'}]
}, required={'next':['question','tools','history']})

### Worked approach

Repeat model action → local observation until a final answer appears or the call budget is used up.

In [ ]:
def run_quote_loop(question, model, tools, tool_descriptions, max_calls=4):
    require(type(max_calls) is int and max_calls > 0, 'Positive call budget required')
    history = []
    for _ in range(max_calls):
        action = ask(model, 'next', question=question, tools=tool_descriptions, history=history)
        if action.get('type') == 'final':
            require(any(row['observation']['ok'] for row in history), 'No successful tool evidence')
            answer = action.get('answer')
            require(isinstance(answer, str) and bool(answer.strip()), 'Missing final answer')
            return {'answer': answer, 'history': history}
        require(action.get('type') == 'tool', 'Unknown action type')
        result = dispatch_tool(action, tools, tool_descriptions)
        history.append({'action': action, 'observation': {'ok': True, 'result': result}})
    raise RuntimeError('Model-call budget exhausted without a final answer')

In [ ]:
model.reset()
result = run_quote_loop(QUESTION, model, TOOLS, TOOL_DESCRIPTIONS)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

The discount lookup observation reaches the next call before quote calculation. These operations have a dependency, so parallel execution would be inappropriate. Tool errors propagate in this simpler loop.

## T3 — Reject a bad tool request and recover

**Pattern:** Tool use · **Time:** 10–15 minutes

The first model request names an unavailable tool. Record a structured error observation without executing it, then allow the model to recover by requesting an allowed lookup. Feed the successful result back before accepting a final answer. Allow at most four model calls. Return `{answer, history}` or raise explicitly if the budget expires.

**Supplied model operations**

- `next`: same action format as T2 and payload `{question, tools, history}`.
- Only `lookup_policy(topic)` is allowed. An unknown tool must produce an error observation listing available tools.
- Do not simulate successful execution of the invalid request.

**Manual success criteria**

- Make three model calls and execute only the single allowed lookup.
- History contains the invalid request, its error observation, the valid lookup and its real result.
- Reject malformed argument keys and record tool exceptions as errors, not factual observations.
- `TOOL_EXECUTIONS` should contain only `returns` after one run; clear it before repeating.

In [ ]:
QUESTION = "What is the return window for unopened accessories?"
TOOL_EXECUTIONS = []
def lookup_policy(topic):
    if topic != 'returns':
        raise ValueError('Unknown topic')
    TOOL_EXECUTIONS.append(topic)
    return {'return_window_days':30,'condition':'unopened accessories'}
TOOLS = {'lookup_policy':lookup_policy}
TOOL_DESCRIPTIONS = {'lookup_policy':{'args':['topic']}}
model = RecordedModel({
 'next':[{'type':'tool','tool':'delete_customer','args':{'customer_id':'C-9'}},
         {'type':'tool','tool':'lookup_policy','args':{'topic':'returns'}},
         {'type':'final','answer':'Unopened accessories have a 30-day return window.'}]
}, required={'next':['question','tools','history']})

### Worked approach

Use the same bounded action/observation loop, but feed validation and tool errors back as explicit error observations.

In [ ]:
def run_policy_loop(question, model, tools, tool_descriptions, max_calls=4):
    require(type(max_calls) is int and max_calls > 0, 'Positive call budget required')
    history = []
    for _ in range(max_calls):
        action = ask(model, 'next', question=question, tools=tool_descriptions, history=history)
        if action.get('type') == 'final':
            require(any(row['observation']['ok'] for row in history), 'No successful tool evidence')
            answer = action.get('answer')
            require(isinstance(answer, str) and bool(answer.strip()), 'Missing final answer')
            return {'answer': answer, 'history': history}
        require(action.get('type') == 'tool', 'Unknown action type')
        try:
            result = dispatch_tool(action, tools, tool_descriptions)
            observation = {'ok': True, 'result': result}
        except Exception as error:
            observation = {'ok': False, 'error': f'{type(error).__name__}: {error}',
                           'available_tools': sorted(tools)}
        history.append({'action': action, 'observation': observation})
    raise RuntimeError('Model-call budget exhausted without a final answer')

In [ ]:
model.reset()
TOOL_EXECUTIONS.clear()
result = run_policy_loop(QUESTION, model, TOOLS, TOOL_DESCRIPTIONS)
print(json.dumps(result, indent=2))
print('Operations:', [call['operation'] for call in model.calls])

### Why it works and what to watch

The unavailable tool never runs. The model receives the error plus the available-tool list, then recovers. A final answer requires at least one successful tool observation; semantic faithfulness still needs review.

# Compare your attempt with the reference

After each exercise, inspect the actual call payloads in `model.calls`. Ask whether each downstream call received the required source data, intermediate results, or error feedback. Compare function behavior and clarity; an equivalent implementation need not match this code line for line.

For concurrent exercises, inspect overlapping call intervals and the timing of synthesis rather than expecting a fixed completion order. For tool loops, inspect history entries to distinguish proposed actions, successful observations, and errors. For refinement, inspect whether the revised output addresses the feedback while preserving correct information.